In [1]:
from data_preparation import data_preparation
from technical_agent import technical_agent
from health_agent import health_agent
from Semantic_functions import FinancialSentimentAnalyser
import numpy as np
import pandas as pd

# Aggregation agent class
class aggregation_agent:
  def __init__(self, company, year, action_threshold=0.2):
        # Initialize with stock company and year
        self.company = company
        self.year = year
        # Define thresholds
        self.act_threshold = action_threshold    # Action threshold for weighted score decision making
        self.conf_threshold = 0.5                # High confidence threshold for conflict condition

  # Function to compute best action from average confidence score
  def get_best_action(self, avg_score):
    if avg_score > self.act_threshold:
      return 1
    elif avg_score < -self.act_threshold:
      return -1
    else:
      return 0

  # Function to compute stock advice using models aggregation
  def get_decision(self, low_weight_agent=None):
    company = self.company
    year = self.year
    conf_threshold = self.conf_threshold
    print(f"\nCompany: {company}\nYear: {year}\nAction threshold: {self.act_threshold}\nConflict high confidence threshold: {conf_threshold}\n")

    # Technical agent Simulation
    print("\n** Technical agent Simulation **")
    technical_sim = technical_agent(company, year)
    df_technical = technical_sim.model_prediction()

    print(f"\nTechnical agent prediction for company '{company}' in year {year}:\n{df_technical}\n")
    print(f"Prediction distribution:\n{df_technical['predict_action'].value_counts()}\n")

    # Health agent Simulation
    print("\n** Health agent Simulation **")
    health_sim = health_agent(company, year)
    df_health = health_sim.model_prediction()

    print(f"\nHealth agent prediction for company '{company}' in year {year}:\n{df_health}\n")
    print(f"Prediction distribution:\n{df_health['predict_action'].value_counts()}\n")

    # Sentiment agent Simulation
    print("\n** Sentiment agent Simulation **")
    sentiment_sim = FinancialSentimentAnalyser()
    df_semantic = sentiment_sim.semantic_year(company, year)
    df_semantic['semantic_pred'] = df_semantic['label'].map({'positive':1, 'neutral':0, 'negative':-1})

    print(f"\nSentiment agent prediction for company '{company}' in year {year}:\n{df_semantic}\n")
    print(f"Prediction distribution:\n{df_semantic['semantic_pred'].value_counts()}\n")

    # Filter and Rename prediction values and confidence scores for each agent
    df_tech_pred = df_technical.rename(
        columns={ 'predict_action': 'technical_pred', 'confidence_level': 'technical_score' }
      )[['technical_pred','technical_score']]
    df_health_pred = df_health.rename(
        columns={ 'predict_action': 'health_pred', 'confidence_level': 'health_score' }
      )[['health_pred','health_score']]
    df_semantic_pred = df_semantic.rename(
        columns={ 'score': 'semantic_score' }
      ).set_index('target_date')[['semantic_pred','semantic_score']]

    # Merge prediction values and confidence scores of all agents
    df_pred = pd.merge(df_tech_pred, df_health_pred, left_index=True, right_index=True)
    df_pred = pd.merge(df_pred, df_semantic_pred, left_index=True, right_index=True, how='left')

    # Prediction and Confidence score columns
    pred_cols = ['technical_pred', 'health_pred', 'semantic_pred']
    score_cols = ['technical_score', 'health_score', 'semantic_score']

    # Condition to reduce weightage of Sentiment agent in overall prediction when enabled
    # Ensure agents equal contribution when sentiment agent with dominant confidence scores
    if low_weight_agent == "sentiment":
      print("\nLow weightage enabled for Sentiment agent:")
      print("Technical agent weight = 100%\nHealth agent weight = 100%\nSentiment agent weight = 80%\n")
      df_pred['semantic_score'] *= 0.8
    df_pred[score_cols] = df_pred[score_cols].round(2)    # Roundoff confidence score by 2 decimal points
    #print(df_pred)

    # Compute normalized weighted average across all agents in range [-1,1]
    # Normalized weighted average = ∑i(prediction[i] * confidence[i]) / ∑i(confidence[i])
    weight_sum = (df_pred[pred_cols].values * df_pred[score_cols].values).sum(axis=1)
    df_pred['weight_avg'] = weight_sum / df_pred[score_cols].sum(axis=1)

    # Conflict condition - Disagreement between all agents with high confidence
    conflict_condition = (((df_pred['technical_pred'] * df_pred['health_pred'] == -1)
                            & ((df_pred['technical_score'] > conf_threshold)
                                & (df_pred['health_score'] > conf_threshold)
                                & (df_pred['semantic_score'] == 0.00))) |
                          (((df_pred[['technical_pred', 'health_pred', 'semantic_pred']].sum(axis=1) == 0)
                                & (df_pred[['technical_pred', 'health_pred', 'semantic_pred']].prod(axis=1) == 0)
                                & (df_pred[['technical_pred', 'health_pred', 'semantic_pred']].abs().sum(axis=1) == 2))
                            & ((df_pred['technical_score'] > conf_threshold)
                                & (df_pred['health_score'] > conf_threshold)
                                & (df_pred['semantic_score'] > conf_threshold))))
    print(f"\nNumber of conflict conditions across agents predictions = {conflict_condition.sum()}\n")
    #print(df_pred[conflict_condition])

    #df_pred['best_pred'] = df_pred['weight_avg'].apply(get_best_action)
    # Compute best prediction through aggregation logic
    # Fix action to 'HOLD' - 0 when conflict conditions
    df_pred['best_pred'] = np.where(conflict_condition, 0, df_pred['weight_avg'].apply(self.get_best_action))

    # Map best action values to decision labels
    df_pred['decision'] = df_pred['best_pred'].map({1:'BUY', 0:'HOLD', -1:'SELL'})
    return df_pred

In [3]:
company = 'BA'
year = 2023

agg_sim = aggregation_agent(company, year, action_threshold=0.1)
df_agg = agg_sim.get_decision(low_weight_agent="sentiment")

print(f"\nAggregation agent prediction for company '{company}' in year {year}:\n{df_agg}\n")
print(f"Prediction distribution:\n{df_agg['decision'].value_counts()}")

pred_map = {1:'BUY', 0:'HOLD', -1:'SELL'}
print("\nHealth agent Prediction distribution:",df_agg['health_pred'].map(pred_map).value_counts())
print("\nTechnial agent Prediction distribution:",df_agg['technical_pred'].map(pred_map).value_counts())
print("\nSentiment agent Prediction distribution:",df_agg['semantic_pred'].map(pred_map).value_counts())

[*********************100%***********************]  1 of 1 completed


Company: BA
Year: 2023
Action threshold: 0.1
Conflict high confidence threshold: 0.5


** Technical agent Simulation **
Class imported successfully!
Class activated!



Technical agent prediction for company 'BA' in year 2023:
            Sell_prob  Hold_prob  Buy_prob  predict_action  confidence_level
Date                                                                        
2023-01-03      0.480      0.148     0.372              -1             0.480
2023-01-04      0.342      0.228     0.430               1             0.430
2023-01-05      0.248      0.226     0.526               1             0.526
2023-01-06      0.410      0.158     0.432               1             0.432
2023-01-09      0.284      0.252     0.464               1             0.464
...               ...        ...       ...             ...               ...
2023-12-22      0.672      0.258     0.070              -1             0.672
2023-12-26      0.696      0.232     0.072              -1             0.696
2023-12-27      0.580      0.340     0.080              -1             0.580
2023-12-28      0.672      0.254     0.074              -1             0.672
2023-12-29      0

[*********************100%***********************]  1 of 1 completed



Health agent prediction for company 'BA' in year 2023:
            Sell_prob  Hold_prob  Buy_prob  predict_action  confidence_level
Date                                                                        
2023-01-03      0.350      0.054     0.596               1             0.596
2023-01-04      0.350      0.054     0.596               1             0.596
2023-01-05      0.350      0.054     0.596               1             0.596
2023-01-06      0.350      0.054     0.596               1             0.596
2023-01-09      0.350      0.054     0.596               1             0.596
...               ...        ...       ...             ...               ...
2023-12-22      0.728      0.108     0.164              -1             0.728
2023-12-26      0.728      0.108     0.164              -1             0.728
2023-12-27      0.728      0.108     0.164              -1             0.728
2023-12-28      0.728      0.108     0.164              -1             0.728
2023-12-29      0.72

Device set to use mps:0



Sentiment agent prediction for company 'BA' in year 2023:
    target_date                                           headline     label  \
0    2023-01-01  US STOCKS-Wall St struggles to stay higher on ...  negative   
1    2023-01-02  US STOCKS-Wall St struggles to stay higher on ...  negative   
2    2023-01-03  US STOCKS-Wall St struggles to stay higher on ...  negative   
3    2023-01-04  US STOCKS-Wall St struggles to stay higher on ...  negative   
4    2023-01-05  US STOCKS-Wall St struggles to stay higher on ...  negative   
..          ...                                                ...       ...   
360  2023-12-27                               No headline in range   neutral   
361  2023-12-28                               No headline in range   neutral   
362  2023-12-29                               No headline in range   neutral   
363  2023-12-30                               No headline in range   neutral   
364  2023-12-31                               No headline in 

In [8]:
company = 'AMD'
year = 2024

agg_sim = aggregation_agent(company, year)
df_agg = agg_sim.get_decision(low_weight_agent="sentiment")

print(f"\nAggregation agent prediction for company '{company}' in year {year}:\n{df_agg}\n")
print(f"Prediction distribution:\n{df_agg['decision'].value_counts()}")

pred_map = {1:'BUY', 0:'HOLD', -1:'SELL'}
print("\nHealth agent Prediction distribution:",df_agg['health_pred'].map(pred_map).value_counts())
print("\nTechnial agent Prediction distribution:",df_agg['technical_pred'].map(pred_map).value_counts())
print("\nSentiment agent Prediction distribution:",df_agg['semantic_pred'].map(pred_map).value_counts())

[*********************100%***********************]  1 of 1 completed


Company: AMD
Year: 2024
Action threshold: 0.2
Conflict high confidence threshold: 0.5


** Technical agent Simulation **
Class imported successfully!
Class activated!



Technical agent prediction for company 'AMD' in year 2024:
            Sell_prob  Hold_prob  Buy_prob  predict_action  confidence_level
Date                                                                        
2024-01-02      0.494      0.270     0.236              -1             0.494
2024-01-03      0.386      0.284     0.330              -1             0.386
2024-01-04      0.310      0.250     0.440               1             0.440
2024-01-05      0.270      0.238     0.492               1             0.492
2024-01-08      0.342      0.264     0.394               1             0.394
...               ...        ...       ...             ...               ...
2024-12-23      0.718      0.056     0.226              -1             0.718
2024-12-24      0.418      0.208     0.374              -1             0.418
2024-12-26      0.482      0.056     0.462              -1             0.482
2024-12-27      0.734      0.054     0.212              -1             0.734
2024-12-30      

[*********************100%***********************]  1 of 1 completed



Health agent prediction for company 'AMD' in year 2024:
            Sell_prob  Hold_prob  Buy_prob  predict_action  confidence_level
Date                                                                        
2024-01-02      0.870      0.004     0.126              -1             0.870
2024-01-03      0.948      0.030     0.022              -1             0.948
2024-01-04      0.968      0.010     0.022              -1             0.968
2024-01-05      0.870      0.004     0.126              -1             0.870
2024-01-08      0.652      0.006     0.342              -1             0.652
...               ...        ...       ...             ...               ...
2024-12-23      0.930      0.000     0.070              -1             0.930
2024-12-24      0.930      0.000     0.070              -1             0.930
2024-12-26      0.930      0.000     0.070              -1             0.930
2024-12-27      0.930      0.000     0.070              -1             0.930
2024-12-30      0.7

Device set to use mps:0



Sentiment agent prediction for company 'AMD' in year 2024:
    target_date                                           headline     label  \
0    2024-01-01                               No headline in range   neutral   
1    2024-01-02                               No headline in range   neutral   
2    2024-01-03                               No headline in range   neutral   
3    2024-01-04                               No headline in range   neutral   
4    2024-01-05                               No headline in range   neutral   
..          ...                                                ...       ...   
361  2024-12-27  S&P Futures Slip Ahead of Key U.S. Retail Sale...  positive   
362  2024-12-28  S&P Futures Slip Ahead of Key U.S. Retail Sale...  positive   
363  2024-12-29  S&P Futures Slip Ahead of Key U.S. Retail Sale...  positive   
364  2024-12-30  S&P Futures Slip Ahead of Key U.S. Retail Sale...  positive   
365  2024-12-31  S&P Futures Slip Ahead of Key U.S. Retail S

In [10]:
company = 'AAPL'
year = 2022

agg_sim = aggregation_agent(company, year)
df_agg = agg_sim.get_decision()

print(f"\nAggregation agent prediction for company '{company}' in year {year}:\n{df_agg}\n")
print(f"Prediction distribution:\n{df_agg['decision'].value_counts()}")

pred_map = {1:'BUY', 0:'HOLD', -1:'SELL'}
print("\nHealth agent Prediction distribution:",df_agg['health_pred'].map(pred_map).value_counts())
print("\nTechnial agent Prediction distribution:",df_agg['technical_pred'].map(pred_map).value_counts())
print("\nSentiment agent Prediction distribution:",df_agg['semantic_pred'].map(pred_map).value_counts())

[*********************100%***********************]  1 of 1 completed


Company: AAPL
Year: 2022
Action threshold: 0.2
Conflict high confidence threshold: 0.5


** Technical agent Simulation **
Class imported successfully!
Class activated!



Technical agent prediction for company 'AAPL' in year 2022:
            Sell_prob  Hold_prob  Buy_prob  predict_action  confidence_level
Date                                                                        
2022-01-03      0.340      0.300     0.360               1             0.360
2022-01-04      0.310      0.370     0.320               0             0.370
2022-01-05      0.258      0.234     0.508               1             0.508
2022-01-06      0.308      0.304     0.388               1             0.388
2022-01-07      0.388      0.166     0.446               1             0.446
...               ...        ...       ...             ...               ...
2022-12-23      0.688      0.138     0.174              -1             0.688
2022-12-27      0.574      0.226     0.200              -1             0.574
2022-12-28      0.566      0.180     0.254              -1             0.566
2022-12-29      0.624      0.176     0.200              -1             0.624
2022-12-30     

[*********************100%***********************]  1 of 1 completed



Health agent prediction for company 'AAPL' in year 2022:
            Sell_prob  Hold_prob  Buy_prob  predict_action  confidence_level
Date                                                                        
2022-01-03      0.382      0.014     0.604               1             0.604
2022-01-04      0.478      0.012     0.510               1             0.510
2022-01-05      0.490      0.012     0.498               1             0.498
2022-01-06      0.480      0.022     0.498               1             0.498
2022-01-07      0.480      0.022     0.498               1             0.498
...               ...        ...       ...             ...               ...
2022-12-23      0.112      0.064     0.824               1             0.824
2022-12-27      0.112      0.064     0.824               1             0.824
2022-12-28      0.112      0.064     0.824               1             0.824
2022-12-29      0.112      0.064     0.824               1             0.824
2022-12-30      0.

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: a680e8e5-1e37-47cd-94a3-0992e96fd9b8)')' thrown while requesting HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json
Retrying in 1s [Retry 1/5].
Device set to use mps:0



Sentiment agent prediction for company 'AAPL' in year 2022:
    target_date                                           headline     label  \
0    2022-01-01  Weekly Preview: Earnings to Watch This Week (A...  positive   
1    2022-01-02  Weekly Preview: Earnings to Watch This Week (A...  positive   
2    2022-01-03  Weekly Preview: Earnings to Watch This Week (A...  positive   
3    2022-01-04  Weekly Preview: Earnings to Watch This Week (A...  positive   
4    2022-01-05  Weekly Preview: Earnings to Watch This Week (A...  positive   
..          ...                                                ...       ...   
360  2022-12-27            Implied MTUM Analyst Target Price: $159   neutral   
361  2022-12-28            Implied MTUM Analyst Target Price: $159   neutral   
362  2022-12-29            Implied MTUM Analyst Target Price: $159   neutral   
363  2022-12-30            Implied MTUM Analyst Target Price: $159   neutral   
364  2022-12-31            Implied MTUM Analyst Target Pric